# 用講的跟 Workflow 對話

示範語音輸出、語音輸入、語音與文字併行輸出，以及回答被中斷後的行為。四項各佔一章，依序執行。

建立需要口語互動的 Agent 時，這四項決定使用者實際感受到的反應方式。

| 章 | 涵蓋的模組 |
| --- | --- |
| 一、文字輸入、語音輸出 | `VoiceAnswerAction` |
| 二、語音輸入 | `VoiceTextPerceive` |
| 三、語音與文字併行輸出 | `VoiceAnswerAction` |
| 四、回答被中斷 | 兩者 |


## 在 Colab 準備環境

安裝 SDK 與其相依套件。

在尚未安裝過的環境執行一次即可。


In [ ]:
!pip install -q "git+https://github.com/R300-AI/Agentic-SDK.git"

## 端點設定

填入轉寫、合成與生成三個端點的位址、金鑰與模型名稱。

四個章節共用這一組設定，且全程連線真實端點，不使用替身物件。


In [ ]:
AZURE_ENDPOINT      = "https://<資源>.cognitiveservices.azure.com"
AZURE_API_KEY       = "<KEY>"
CHAT_BASE_URL       = "https://<資源>/openai/v1/"
CHAT_API_KEY        = "<KEY>"
CHAT_MODEL          = "<部署名稱>"
TRANSCRIBE_MODEL    = "<部署名稱>"
TTS_MODEL           = "<部署名稱>"
REALTIME_API_VERSION = "2025-04-01-preview"
SPEECH_API_VERSION   = "2025-03-01-preview"

## 連線方式不同的端點

覆寫傳輸開啟連線的方法，接上非 OpenAI 的端點。

端點的位址規則、認證方式或查詢參數與 OpenAI 不同時執行本節；使用 OpenAI 端點時整節略過，直接以 `RealtimeTranscription(api_key=..., model=...)` 建立傳輸。

- SDK 內附的傳輸僅建立 `OpenAI` client，不內建其他供應商的整合。
- 覆寫的範圍僅限開啟連線。session 設定、音訊傳送、事件分派、回合判定與靜音閘門由基底類別提供。
- `turn_detection` 屬於傳輸的參數。它決定服務依固定的靜音長度判定語句結束，或由模型判斷該停頓屬於句中換氣或句末。


In [ ]:
from openai import AzureOpenAI
from agentic_sdk.audio.realtime import RealtimeTranscription
from agentic_sdk.audio.speech import SpeechOutput


class MyTranscription(RealtimeTranscription):
    """我自己接的端點。SDK 只認 OpenAI，這一家是我接的。"""

    def _open(self):
        return AzureOpenAI(azure_endpoint=AZURE_ENDPOINT, api_key=AZURE_API_KEY,
                           api_version=REALTIME_API_VERSION).beta.realtime.connect(
            model=self._model, extra_query={"intent": "transcription"})


class MySpeech(SpeechOutput):
    def _open_stream(self, text):
        return AzureOpenAI(azure_endpoint=AZURE_ENDPOINT, api_key=AZURE_API_KEY,
                           api_version=SPEECH_API_VERSION
                           ).audio.speech.with_streaming_response.create(
            model=self._model, voice=self._voice, input=text,
            response_format=self._response_format)

## 一、文字輸入、語音輸出

以鍵盤輸入問題，並由 `VoiceAnswerAction` 同時產出語音與畫面兩個頻道。

需要語音回覆但不需要語音輸入時採用這個組合，perceive 使用一般的 `PassThroughPerceive`。

| 頻道 | 內容 |
| --- | --- |
| `spoken` | 口語，承載判斷與理由 |
| `displayed` | 視覺內容，承載條列、型號與數值 |

兩個頻道互為補充，非同一份內容的重複。


In [ ]:
from agentic_sdk import Workflow
from agentic_sdk.modules import PassThroughPerceive, VoiceAnswerAction

class Recorded(MySpeech):
    """把合成出來的音訊留下來，這樣 Colab 可以播給你聽。"""
    def __init__(self, **kw):
        self.audio = bytearray()
        self.said = []
        super().__init__(**kw)
    def speak(self, text):
        self.said.append(text)
        for piece in super().speak(text):
            self.audio.extend(piece)
            yield piece

speaking = Recorded(model=TTS_MODEL, voice="alloy")
action = VoiceAnswerAction(speech=speaking, api_key=CHAT_API_KEY,
                           base_url=CHAT_BASE_URL, model=CHAT_MODEL)
workflow = Workflow(workflow_name="打字問、聲音答",
                    perceive=PassThroughPerceive(), action=action)
result = workflow.run("保固多久？我要能貼在說明頁上的版本。")

print("畫面上顯示 :", result.final_message.replace("\n", " / ")[:80])
print("說出口的   :", speaking.said[0][:60])
print("合成音訊   :", len(speaking.audio), "bytes")

## 二、語音輸入

以合成音訊模擬麥克風，並由 `VoiceTextPerceive` 將語音轉為該回合的輸入。

在沒有音訊裝置的環境中示範語音輸入路徑時執行本節。

- 低於音量門檻的片段不會離開本機。純靜音與語音的計費相同，且服務會將其辨識為未曾說出的字詞。
- 語句結束後仍需送出一段靜音。服務依據靜音判定語句結束，音訊在說完的瞬間中斷則轉寫不會返回。
- `run()` 不需再次取得使用者的輸入內容。語音抵達的時機取決於使用者，模組先行收取，工作流在執行時取用。


In [ ]:
import array, time
from agentic_sdk.modules import VoiceTextPerceive

def as_microphone(text, *, quiet_after=2.5, rate_in=24000, rate_out=16000):
    """用合成語音假裝有人在講話，這樣沒有麥克風也能示範。

    尾巴要補靜音。真的麥克風在人停止說話之後還是持續在收，而服務就是靠
    「聽到靜音」判定一句話結束——講完就把音訊切掉，轉寫永遠不會回來。
    """
    pcm = b"".join(MySpeech(model=TTS_MODEL).speak(text))
    samples = array.array("h"); samples.frombytes(pcm)
    step = rate_in / rate_out
    out = array.array("h", (samples[int(i * step)] for i in range(int(len(samples) / step))))
    out.extend([0] * int(rate_out * quiet_after))
    frame = 1600                       # 十分之一秒
    return [out[i:i + frame].tobytes() for i in range(0, len(out), frame)]

listening = MyTranscription(model=TRANSCRIBE_MODEL, language="zh",
                            turn_detection={"type": "semantic_vad", "eagerness": "low"})
perceive = VoiceTextPerceive(transport=listening, speech_threshold=500, hangover_seconds=1.2)

# 從轉寫回呼旁觀就好。pending_input() 是「取用」——印出來就等於用掉了，
# 待會 run() 會拿不到東西。
heard = []
listening.on_transcript(heard.append)

chunks = as_microphone("保固期是多久？")
for chunk in chunks:
    perceive.hear(chunk)               # 安靜的片段不會離開這台機器
    time.sleep(0.1)                    # 麥克風是即時來的
for _ in range(60):                    # 等服務判定這句話講完
    if perceive.pending_input():
        break
    time.sleep(0.2)

print("麥克風片段數 :", len(chunks))
print("聽到的話     :", heard[0] if heard else "（還沒回來）")

speaking = Recorded(model=TTS_MODEL, voice="alloy")
action = VoiceAnswerAction(speech=speaking, api_key=CHAT_API_KEY,
                           base_url=CHAT_BASE_URL, model=CHAT_MODEL)
talking = Workflow(workflow_name="用講的問、用聲音答", perceive=perceive, action=action)
result = talking.run()                 # 不必再告訴它使用者說了什麼

print("畫面上顯示   :", result.final_message.replace("\n", " / ")[:60])
print("說出口的     :", speaking.said[0][:50])

## 三、語音與文字併行輸出

量測語音開始輸出與文字產生完畢之間的時間差。

回覆較長時，這個差距決定使用者等待多久才聽到第一句。

`spoken` 欄位完成時即送往合成，不等待整段回覆產生完畢。


In [ ]:
written = []
started = []

class Timed(Recorded):
    """記下「開始說話」是這一輪的第幾秒。"""
    def speak(self, text):
        started.append(time.monotonic() - began)
        return super().speak(text)

speaking = Timed(model=TTS_MODEL, voice="alloy")
action = VoiceAnswerAction(speech=speaking, api_key=CHAT_API_KEY,
                           base_url=CHAT_BASE_URL, model=CHAT_MODEL)
workflow = Workflow(workflow_name="邊說邊顯示",
                    perceive=PassThroughPerceive(), action=action)

began = time.monotonic()
stream = workflow.stream("保固多久？請給我可以貼在說明頁上的完整條列。")
for piece in stream:
    written.append(piece)
finished = time.monotonic() - began

print(f"開始說話 : 第 {started[0]:.1f} 秒")
print(f"整段寫完 : 第 {finished:.1f} 秒（共 {len(''.join(written))} 個字）")
print(f"→ 聲音比整段文字早了 {finished - started[0]:.1f} 秒")

## 四、回答被中斷

在回答進行中送入使用者的語音，並檢視中斷後保留的回合記錄。

使用者插話時，這個行為決定下一回合的上下文包含哪些內容。

- 中斷的觸發依據為語音活動，而非轉寫內容。語音活動約於開口後 600 毫秒可偵測，轉寫需接近四秒。
- 對話記錄保留實際送達使用者的內容。未送達的部分不進入下一回合，以避免 Agent 引用使用者未曾接收的內容。
- 被打斷有自己的 `stop_reason` 值，不算錯誤。該旗標用於流程自我中止並以錯誤形式呈現，使用者主動插話不屬於錯誤。


In [ ]:
from agentic_sdk.core.cancellation import CancellationToken

listening = MyTranscription(model=TRANSCRIBE_MODEL, language="zh",
                            turn_detection={"type": "semantic_vad", "eagerness": "low"})
perceive = VoiceTextPerceive(transport=listening, speech_threshold=500, hangover_seconds=1.2)

class Interrupted(Recorded):
    """一邊播放，一邊讓使用者在中途開口。"""
    def speak(self, text):
        for index, piece in enumerate(super().speak(text)):
            if index == 3:                        # 播到一小段就插話
                for chunk in barge_in:
                    perceive.hear(chunk)
            yield piece

barge_in = as_microphone("等一下，我不是問這個。", quiet_after=1.0)

speaking = Interrupted(model=TTS_MODEL, voice="alloy")
action = VoiceAnswerAction(speech=speaking, api_key=CHAT_API_KEY,
                           base_url=CHAT_BASE_URL, model=CHAT_MODEL)
talking = Workflow(workflow_name="會被打斷的對話", perceive=perceive, action=action)

for chunk in as_microphone("保固期是多久？"):
    perceive.hear(chunk)
    time.sleep(0.1)
for _ in range(60):
    if perceive.pending_input():
        break
    time.sleep(0.2)

cut = talking.run(cancel=CancellationToken())
print("為什麼停下來 :", cut.stop_reason)
print("這是錯誤嗎 :", cut.stop_reason not in ("end_turn", "interrupted"))
print("對方收到的 :", (cut.interrupt_payload.get("delivered") or "")[:40])
print("記憶裡留的 :", talking.memory.turns[-1].content[:40])

## 音訊裝置的責任歸屬

將本文件的替代實作換成實際的音訊裝置。

SDK 不擷取麥克風也不播放音訊，兩端由呼叫端接續。

| 方向 | 接續方式 | 本文件的替代實作 |
| --- | --- | --- |
| 輸入 | 將音訊片段送入 `hear()` | `as_microphone` 以合成音訊產生片段 |
| 輸出 | 在 `speak()` 外包一層，每段交給裝置後再 `yield` | `Recorded` 保存音訊而非播放 |

輸出側的順序具有意義。先播放再 `yield`，中斷時放棄該串流才能同時停止播放與合成。

完整且可執行的範例位於 `examples/voice/desktop_voice_agent.py`，以 WAV 檔案作為音訊來源。
